In [ ]:
import pandas as pd
from time import time
import InsurAutoML
from InsurAutoML import load_data, train_test_split
from InsurAutoML.hpo.informed.fixed import InformedAutoTabular
# from InsurAutoML import InformedAutoTabular

seed = 42
n_trials = 64
N_ESTIMATORS = 4
TIMEOUT = (n_trials / 4) * 450

InsurAutoML.set_seed(seed)

In [ ]:
# load data
database = load_data(data_type = ".csv").load(path = "")
database_names = [*database]
database_names

In [ ]:
database["ausprivauto"].head(5)

In [ ]:
# define response/features
response = "ClaimAmount"
features = list(
    set(database["ausprivauto"].columns) - set(["ClaimOcc", "ClaimNb", "ClaimAmount"])
)
features.sort()

In [ ]:
# train/test split
# first time running
train_X, test_X, train_y, test_y = train_test_split(
    database['ausprivauto'][features], database['ausprivauto'][[response]], test_perc = 0.1, seed = seed
)
train_y = train_y[response]
test_y = test_y[response]
pd.DataFrame(train_X.index.sort_values()).to_csv("train_index.csv", index=False)
# Use the same train/test split across all models for 2+ runs
# train_idx = pd.read_csv("train_index.csv", header=None).values.flatten()
# test_idx = database["ausprivauto"].index.difference(train_idx)
# train_X, test_X, train_y, test_y = (
#     database["ausprivauto"].loc[train_idx, features],
#     database["ausprivauto"].loc[test_idx, features],
#     database["ausprivauto"].loc[train_idx, response],
#     database["ausprivauto"].loc[test_idx, response],
# )

In [ ]:
# fit AutoML model
mol = InformedAutoTabular(
    model_name="ausprivauto_informed_{}".format(n_trials),
    max_evals=n_trials,
    n_estimators=N_ESTIMATORS,
    timeout=TIMEOUT,
    search_algo="Optuna",
    objective="MSE",
    cpu_threads=12,
    exclude={"model": ["GaussianProcess", "SGD"]},
    seed=seed,
)
start_time = time()
mol.fit(train_X, train_y)
end_time = time()

In [ ]:
from sklearn.metrics import mean_squared_error
y_train_pred = mol.predict(train_X)
y_test_pred = mol.predict(test_X)
print("Trial number: ", n_trials)
print("Training time (s): ", end_time - start_time)
print("Train MSE: ", mean_squared_error(train_y, y_train_pred))
print("Test MSE: ", mean_squared_error(test_y, y_test_pred))